# ARIS: End-to-End Tutorial and Workflow
### Differentiable Analysis-by-Synthesis for Voice Research

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/N1r/ARIS_nsf/blob/main/notebooks/ARIS_Tutorial_and_Workflow.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-ARIS__nsf-181717.svg)](https://github.com/N1r/ARIS_nsf)

This notebook provides a self-contained walkthrough of ARIS (Analytic Resonance for Interpretable Synthesis):
1. **Environment Setup & Diagnostics**: verifying GPU and dependencies
2. **Audio Inspection & F0 Tracking**: acoustic feature extraction using WORLD
3. **Dataset Manifest Preparation**: normalization and train/val/test splitting
4. **Model Training**: training the differentiable `aria-golf` neural source-filter vocoder
5. **Acoustic Manipulation & Resynthesis**: targeted formant ($F_1$), pitch ($F_0$), and glottal source ($R_d$) modifications
6. **Interactive Studio**: launching the browser-based workspace for continuous parameter exploration


## 1. Setup and Environment

Verify GPU availability, clone the repository if running on Colab, and install required dependencies.


In [ ]:
!nvidia-smi


In [ ]:
import os
import sys

# Configure environment and project path
if not os.path.exists("pyproject.toml"):
    !git clone https://github.com/N1r/ARIS_nsf.git
    %cd ARIS_nsf

src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

!pip install -q -e ".[all]"


In [ ]:
import aris

# Run system diagnostic checks
for item in aris.doctor():
    status = "OK" if item.ok else "--"
    print(f"[{status:2}] {item.name:12} : {item.detail}")


## 2. Audio Data and Acoustic Features

Download the reference Mandarin female speaker dataset (`demo_f024`, 16 kHz), load a sample utterance, and extract its fundamental frequency ($F_0$) trajectory using the WORLD vocoder.


In [ ]:
import urllib.request
import zipfile
from pathlib import Path

# Fetch reference demo package if not present
archive_path = Path("aris_f024_demo.zip")
if not Path("demo_f024").exists():
    url = "https://github.com/N1r/ARIS_nsf/releases/download/v0.1.0/aris_f024_demo.zip"
    urllib.request.urlretrieve(url, archive_path)
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(".")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

from aris.audio import estimate_f0, read_audio

# Inspect sample utterance
sample_path = "demo_f024/dataset/audio/F024_ci1-29ba87b716.wav"
audio, sr = read_audio(sample_path)
f0, backend = estimate_f0(audio, sr, floor_hz=50, ceiling_hz=800, method="pyworld")
time_axis = np.linspace(0, len(audio) / sr, len(f0))

print(f"Sampling rate: {sr} Hz, Duration: {len(audio)/sr:.2f} s, Backend: {backend}")
display(Audio(audio, rate=sr))

# Plot waveform and F0 trajectory
fig, (ax_wave, ax_f0) = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
ax_wave.plot(np.linspace(0, len(audio) / sr, len(audio)), audio, color="#333333", linewidth=0.8)
ax_wave.set_ylabel("Amplitude")
ax_wave.set_title("Waveform")

f0_voiced = np.where(f0 > 0, f0, np.nan)
ax_f0.plot(time_axis, f0_voiced, color="#1b7837", linewidth=2.0)
ax_f0.set_ylabel("F0 (Hz)")
ax_f0.set_xlabel("Time (s)")
ax_f0.set_ylim(50, 400)
ax_f0.set_title("Fundamental Frequency (F0)")
plt.tight_layout()
plt.show()


## 3. Dataset Preparation

Process raw audio into a standardized experiment directory: resample to 16 kHz, precompute $F_0$ features, and partition into train/validation/test splits.


In [ ]:
# Build manifest and extract acoustic features across the corpus
manifest = aris.prepare(
    source="demo_f024/dataset/audio",
    output="data/prepared_corpus",
    sample_rate=16000,
    f0_method="pyworld",
    min_duration=0.5,
    overwrite=True,
)

print(f"Indexed {len(manifest.records)} utterances.")
errors = aris.validate("data/prepared_corpus")
if errors:
    print("Validation warnings:", errors)
else:
    print("Dataset validation passed.")


## 4. Model Training

Configure and train an `aria-golf` vocoder on the prepared dataset. `aria-golf` models source-filter interaction using differentiable time-varying LPC and glottal flow wavetables, enabling independent control over vocal tract resonances and glottal pulse parameters.


In [ ]:
# Initialize experiment configuration
exp_dir = aris.init_experiment(
    dataset="data/prepared_corpus",
    output="experiments/tutorial_run",
    model="aria-golf",
    batch_size=16,
    max_steps=50,
    overwrite=True,
)

# Execute training iteration (50 steps for pipeline demonstration)
aris.train(exp_dir, extra_args=["--trainer.max_steps=50"])


## 5. Resynthesis and Acoustic Manipulation

Using the converged reference checkpoint, we resynthesize held-out evaluation audio and generate controlled experimental stimuli along targeted phonetic dimensions:
- `f1_scale`: scaling factor applied to the first formant frequency (vowel height / aperture)
- `pitch_semitones`: uniform semitone shift applied to the voiced pitch contour
- `glottal_rd_scale`: glottal flow shape parameter (breathy vs. creaky phonation)


In [ ]:
from pathlib import Path

checkpoint = "demo_f024/experiment/runs/checkpoints/last.ckpt"
recon_dir = Path("out/reconstruction")
stimuli_dir = Path("out/stimuli")

# 1. Baseline Resynthesis (synthesizes held-out test utterances)
aris.synthesize("demo_f024/experiment", checkpoint, recon_dir, overwrite=True)

# 2. Targeted Acoustic Manipulation
aris.manipulate(
    "demo_f024/experiment",
    checkpoint,
    stimuli_dir,
    variants=[
        "f1_up:f1_scale=1.2",
        "pitch_down:pitch_semitones=-4",
        "breathy:glottal_rd_scale=1.6",
        "creaky:glottal_rd_scale=0.6",
    ],
    overwrite=True,
)


In [ ]:
# Auditory Comparison
stem = "F024_bian4"
orig_wav = Path("demo_f024/dataset/audio") / f"{stem}-2e74da0c07.wav"
recon_wav = recon_dir / f"{stem}-2e74da0c07.wav"
f1_wav = stimuli_dir / "f1_up" / recon_wav.name
pitch_wav = stimuli_dir / "pitch_down" / recon_wav.name
breathy_wav = stimuli_dir / "breathy" / recon_wav.name
creaky_wav = stimuli_dir / "creaky" / recon_wav.name

print("Original Recording:")
display(Audio(filename=str(orig_wav)))

print("Reconstructed Baseline:")
display(Audio(filename=str(recon_wav)))

print("F1 Shift (+20%):")
display(Audio(filename=str(f1_wav)))

print("Pitch Shift (-4 Semitones):")
display(Audio(filename=str(pitch_wav)))

print("Breathy Voice (Rd = 1.6):")
display(Audio(filename=str(breathy_wav)))

print("Creaky Voice (Rd = 0.6):")
display(Audio(filename=str(creaky_wav)))


In [ ]:
# Auditory Comparison
stem = "F024_bian4"
orig_wav = sorted(Path("demo_f024/dataset/audio").glob(f"{stem}*.wav"))[0]
recon_wav = sorted(recon_dir.glob(f"{stem}*.wav"))[0]
f1_wav = sorted((stimuli_dir / "f1_up").glob(f"{stem}*.wav"))[0]
pitch_wav = sorted((stimuli_dir / "pitch_down").glob(f"{stem}*.wav"))[0]
breathy_wav = sorted((stimuli_dir / "breathy").glob(f"{stem}*.wav"))[0]
creaky_wav = sorted((stimuli_dir / "creaky").glob(f"{stem}*.wav"))[0]

print("Original Recording:")
display(Audio(filename=str(orig_wav)))

print("Reconstructed Baseline:")
display(Audio(filename=str(recon_wav)))

print("F1 Shift (+20%):")
display(Audio(filename=str(f1_wav)))

print("Pitch Shift (-4 Semitones):")
display(Audio(filename=str(pitch_wav)))

print("Breathy Voice (Rd = 1.6):")
display(Audio(filename=str(breathy_wav)))

print("Creaky Voice (Rd = 0.6):")
display(Audio(filename=str(creaky_wav)))


## 6. Interactive Studio

Launch the browser-based workbench for continuous slider adjustment and A/B listening.


In [ ]:
# Launch Gradio interface (share=True creates a public URL)
aris.launch_studio(workspace=".", share=True, open_browser=False)


## 7. Export Stimuli

Package generated stimuli and computational provenance metadata (`manipulation.json`) for experimental deployment.


In [ ]:
import shutil

shutil.make_archive("aris_stimuli", "zip", "out/stimuli")
print("Saved stimulus package: aris_stimuli.zip")
# from google.colab import files
# files.download("aris_stimuli.zip")
